# 18.1 OS 系统 AI 运行时与共享基座

模拟系统 daemon：共享基座 + 多 App 适配器路由 + 内存压力下的卸载。

In [ ]:
import hashlib
import json
import math
import time
from dataclasses import dataclass, field
from typing import Optional
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    torch.manual_seed(42)
    print(f"PyTorch {torch.__version__}")
except Exception as e:
    torch = None
    print("torch unavailable:", e)

np.random.seed(42)

In [ ]:
@dataclass
class Adapter:
    app_id: str
    name: str
    size_mb: float
    last_used: float = 0.0


class SystemAIRuntime:
    def __init__(self, base_mb=1100, budget_mb=1800):
        self.base_mb = base_mb
        self.budget = budget_mb
        self.adapters: dict[str, Adapter] = {}
        self.foreground: Optional[str] = None

    @property
    def used(self):
        return self.base_mb + sum(a.size_mb for a in self.adapters.values())

    def focus(self, app_id: str):
        self.foreground = app_id
        if app_id in self.adapters:
            self.adapters[app_id].last_used = time.perf_counter()

    def ensure_adapter(self, app_id: str, name: str, size_mb: float):
        if app_id in self.adapters:
            self.focus(app_id)
            return []
        evicted = []
        while self.used + size_mb > self.budget:
            # 不驱逐前台
            cands = [a for a in self.adapters.values() if a.app_id != self.foreground]
            if not cands:
                raise MemoryError("cannot load adapter under budget")
            vic = min(cands, key=lambda a: a.last_used)
            del self.adapters[vic.app_id]
            evicted.append(vic.app_id)
        self.adapters[app_id] = Adapter(app_id, name, size_mb, time.perf_counter())
        self.focus(app_id)
        return evicted

    def infer(self, app_id: str, prompt: str) -> str:
        ad = self.adapters.get(app_id)
        skill = ad.name if ad else "base"
        return f"[{skill}] reply_to={prompt!r}"


rt = SystemAIRuntime(base_mb=1100, budget_mb=1600)
print("load translator", rt.ensure_adapter("app.trans", "lora-translate", 40))
print("load writer", rt.ensure_adapter("app.write", "lora-write", 45))
rt.focus("app.trans")
print("load code (may evict writer)", rt.ensure_adapter("app.code", "lora-code", 420))
print("adapters", list(rt.adapters), "used", rt.used)
print(rt.infer("app.trans", "hello"))
print(rt.infer("app.unknown", "hi"))

## 小结

系统只常驻基座；适配器按前台与 LRU 管理；App 经 daemon 推断，不各自嵌满血模型。